# 60 — SCENIC+ Consensus eRegulons

For each species, identifies eRegulon triplets (TF, Region, Gene, regulation) detected in all seed runs + non-seed run.
Saves results with threshold naming: `all{N}runs`, `90pct`, `80pct`.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter

SCENICPLUS_DIR = Path("/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/scenicplus")
OUT_DIR = SCENICPLUS_DIR / "consensus_eregulons"
OUT_DIR.mkdir(exist_ok=True)

SPECIES = ["Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset"]
SEEDS   = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,42,123,666]
TRIPLET_COLS = ["TF", "Region", "Gene", "regulation"]
SCORE_COLS   = ["importance_R2G", "rho_R2G", "importance_TF2G", "rho_TF2G",
                "importance_x_rho", "triplet_rank"]

print(f"Output dir: {OUT_DIR}")

Output dir: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/scenicplus/consensus_eregulons


In [2]:
def load_ereg(path):
    f = path / "eRegulons_extended.tsv"
    return pd.read_csv(f, sep="\t") if f.exists() else None

def make_triplet_key(df):
    return list(zip(df["TF"], df["Region"], df["Gene"], df["regulation"]))

summary_rows = []

for species in SPECIES:
    runs = {}  # label -> DataFrame

    # Non-seed run
    ns_dir = SCENICPLUS_DIR / f"scplus_pipeline_{species}/Snakemake"
    df_ns = load_ereg(ns_dir)
    if df_ns is not None:
        runs["non_seed"] = df_ns

    # Seed runs
    for seed in SEEDS:
        s_dir = SCENICPLUS_DIR / f"scplus_pipeline_{species}_seed{seed}_dsMin/Snakemake"
        df_s = load_ereg(s_dir)
        if df_s is not None:
            runs[f"seed{seed}"] = df_s

    n_runs = len(runs)
    all_dfs = list(runs.values())
    print(f"\n{species}: {n_runs} runs loaded")

    # Count how many runs contain each triplet
    triplet_counts = Counter()
    for df in all_dfs:
        for t in set(make_triplet_key(df)):
            triplet_counts[t] += 1

    # Thresholds
    thresholds = {
        f"all{n_runs}runs": n_runs,
        "90pct": max(1, round(0.9 * n_runs)),
        "80pct": max(1, round(0.8 * n_runs)),
        "50pct": max(1, round(0.5 * n_runs)),
    }

    # Base DataFrame: non-seed (or first available) with score columns
    base_df = all_dfs[0].copy()
    base_df["_tkey"] = make_triplet_key(base_df)
    base_df["n_runs_detected"] = base_df["_tkey"].map(triplet_counts)
    base_df["n_runs_total"]    = n_runs
    base_df["pct_runs_detected"] = base_df["n_runs_detected"] / n_runs

    # Compute median scores across all runs for each triplet
    # Build a long table, merge, compute median
    long_rows = []
    for run_label, df in runs.items():
        df2 = df[TRIPLET_COLS + [c for c in SCORE_COLS if c in df.columns]].copy()
        df2["run"] = run_label
        long_rows.append(df2)
    long_df = pd.concat(long_rows, ignore_index=True)
    available_scores = [c for c in SCORE_COLS if c in long_df.columns]
    median_scores = (long_df.groupby(TRIPLET_COLS)[available_scores]
                    .median().reset_index()
                    .rename(columns={c: f"median_{c}" for c in available_scores}))

    base_df = base_df.merge(
        median_scores, on=TRIPLET_COLS, how="left", suffixes=("_nonseed","")
    )

    for thresh_name, min_runs in thresholds.items():
        consensus_df = base_df[base_df["n_runs_detected"] >= min_runs].copy()
        consensus_df = consensus_df.drop(columns=["_tkey"], errors="ignore")
        consensus_df = consensus_df.sort_values("n_runs_detected", ascending=False)

        out_file = OUT_DIR / f"eRegulons_consensus_{species}_{thresh_name}.tsv"
        consensus_df.to_csv(out_file, sep="\t", index=False)
        print(f"  [{thresh_name:>14}] {len(consensus_df):>7,} triplets  → {out_file.name}")

        summary_rows.append({
            "species": species, "threshold": thresh_name,
            "min_runs": min_runs, "n_runs_total": n_runs,
            "n_triplets": len(consensus_df),
            "n_TFs": consensus_df["TF"].nunique(),
            "n_genes": consensus_df["Gene"].nunique(),
            "n_regions": consensus_df["Region"].nunique(),
        })

    # Also save eRegulon-level (TF + regulation) consensus
    ereg_counts = Counter()
    for df in all_dfs:
        for t in set(zip(df["TF"], df["regulation"])):
            ereg_counts[t] += 1
    ereg_summary = (pd.DataFrame([{"TF": t, "regulation": r, "n_runs": c, "n_runs_total": n_runs,
                                    "pct_runs": c/n_runs}
                                   for (t,r), c in ereg_counts.items()])
                    .sort_values("n_runs", ascending=False))
    ereg_file = OUT_DIR / f"eRegulon_TF_level_{species}.tsv"
    ereg_summary.to_csv(ereg_file, sep="\t", index=False)
    print(f"  TF-level eRegulon file: {ereg_file.name} ({len(ereg_summary)} TF×regulation combos)")

print("\nDone.")


Human: 21 runs loaded


  [     all21runs]  10,564 triplets  → eRegulons_consensus_Human_all21runs.tsv


  [         90pct]  21,329 triplets  → eRegulons_consensus_Human_90pct.tsv


  [         80pct]  29,144 triplets  → eRegulons_consensus_Human_80pct.tsv


  [         50pct]  54,861 triplets  → eRegulons_consensus_Human_50pct.tsv


...


Marmoset: 21 runs loaded


  [     all21runs]   8,128 triplets  → eRegulons_consensus_Marmoset_all21runs.tsv


  [         90pct]  15,369 triplets  → eRegulons_consensus_Marmoset_90pct.tsv


  [         80pct]  20,614 triplets  → eRegulons_consensus_Marmoset_80pct.tsv


  [         50pct]  36,687 triplets  → eRegulons_consensus_Marmoset_50pct.tsv
  TF-level eRegulon file: eRegulon_TF_level_Marmoset.tsv (142 TF×regulation combos)

Done.


In [3]:
# Summary table across all species and thresholds
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(OUT_DIR / "consensus_summary_all_species.csv", index=False)
print(f"\nSummary saved to: {OUT_DIR / 'consensus_summary_all_species.csv'}")

   species threshold  min_runs  n_runs_total  n_triplets  n_TFs  n_genes  n_regions
     Human all21runs        21            21       10564     41     1879       7339
     Human     90pct        19            21       21329     64     3114      13271
     Human     80pct        17            21       29144     78     3808      17152
     Human     50pct        10            21       54861    104     5619      28544
    Bonobo all21runs        21            21       12830     37     3878       9009
    Bonobo     90pct        19            21       21800     42     5117      13683
    Bonobo     80pct        17            21       27495     44     5731      16248
    Bonobo     50pct        10            21       42661     50     6894      22460
Chimpanzee all21runs        21            21       10862     30     2858       8473
Chimpanzee     90pct        19            21       23362     39     4625      16052
Chimpanzee     80pct        17            21       31902     42     5439    

In [4]:
# Cross-species conserved triplets: present in consensus of ALL 6 species
# Use 90pct threshold per species as input
dfs_90 = {}
for species in SPECIES:
    f = OUT_DIR / f"eRegulons_consensus_{species}_90pct.tsv"
    if f.exists():
        dfs_90[species] = pd.read_csv(f, sep="\t")

if len(dfs_90) == 6:
    # Triplets that have an ortholog in all species — requires mapping to peak_ids
    # For now: find (TF, Gene) pairs conserved across all species (region-agnostic)
    tf_gene_sets = {sp: set(zip(df["TF"], df["Gene"])) for sp, df in dfs_90.items()}
    pan_primate = tf_gene_sets[SPECIES[0]]
    for sp in SPECIES[1:]:
        pan_primate = pan_primate & tf_gene_sets[sp]
    print(f"Pan-primate (TF, Gene) pairs in 90pct consensus of all 6 species: {len(pan_primate)}")

    # Save
    pan_rows = [{"TF": tf, "Gene": g} for (tf, g) in pan_primate]
    pd.DataFrame(pan_rows).to_csv(OUT_DIR / "pan_primate_TF_Gene_pairs_90pct.tsv", sep="\t", index=False)
    print("Saved pan_primate_TF_Gene_pairs_90pct.tsv")

Pan-primate (TF, Gene) pairs in 90pct consensus of all 6 species: 138
Saved pan_primate_TF_Gene_pairs_90pct.tsv
